# Train the ACDC object detector in Google Colab

This notebook reads the official ACDC ZIP files from Google Drive, extracts only adverse-condition train/validation images onto Colab's temporary disk, converts them with this repository's COCO-to-YOLO converter, trains YOLOv8, and copies the best checkpoint and key results back to Drive. The source ZIPs and trained weights are not committed to Git.

In Colab, select **Runtime → Change runtime type → T4 GPU** (or another GPU) before running. ACDC is licensed for non-commercial use; follow its license and citation requirements.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Set the Drive paths

The paths below match the folders you provided under **My Drive**. If the ACDC folder is in a Shared Drive, change `DRIVE_DATA_ROOT` to its mounted path. The folder should contain `gt_detection/gt_detection_trainval.zip` and `rgb_anon/rgb_anon_trainvaltest.zip`.

In [ ]:
from pathlib import Path
import codecs
import os
import shutil
import subprocess
import sys

DRIVE_DATA_ROOT = Path('/content/drive/MyDrive/datasets/ACDC')
ANNOTATIONS_ZIP = DRIVE_DATA_ROOT / 'gt_detection' / 'gt_detection_trainval.zip'
IMAGES_ZIP = DRIVE_DATA_ROOT / 'rgb_anon' / 'rgb_anon_trainvaltest.zip'
DRIVE_OUTPUT_DIR = DRIVE_DATA_ROOT / 'training_output'
DRIVE_CHECKPOINT_DIR = DRIVE_OUTPUT_DIR / 'checkpoints'
TRAINING_COMPLETE_MARKER = DRIVE_OUTPUT_DIR / 'training_complete.txt'

PROJECT_DIR = Path('/content/ETH_AI_OBJ_DETECTION')
ACDC_ROOT = Path('/content/acdc_source')
PROCESSED_DIR = Path('/content/acdc_processed')
REPO_URL = 'https://github.com/Danielit707/ETH_AI_OBJ_DETECTION.git'

for archive in (ANNOTATIONS_ZIP, IMAGES_ZIP):
    if not archive.is_file():
        raise FileNotFoundError(
            f'Missing {archive}. Check DRIVE_DATA_ROOT and confirm the ZIP is uploaded to Drive.'
        )

if not PROJECT_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(PROJECT_DIR)], check=True)
elif not (PROJECT_DIR / 'dataset' / 'convert_coco_to_yolo.py').is_file():
    raise FileNotFoundError(f'{PROJECT_DIR} exists but does not contain this project checkout.')
else:
    subprocess.run(['git', '-C', str(PROJECT_DIR), 'pull', '--ff-only'], check=True)

try:
    import ultralytics  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'ultralytics'], check=True)

gpu_check = subprocess.run(
    [sys.executable, '-c', 'import torch; print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU detected")'],
    check=True,
    capture_output=True,
    text=True,
)
print('Compute device:', gpu_check.stdout.strip())
print('Annotation archive:', ANNOTATIONS_ZIP)
print('Image archive:', IMAGES_ZIP)
print('Colab disk space:', shutil.disk_usage('/content').free // (1024 ** 3), 'GiB free')

## Extract train/validation data to Colab disk

Only `train` and `val` images are extracted from the RGB ZIP; test images and normal-condition reference images are skipped. Extraction to `/content` avoids training directly from Drive, which is typically much slower. Colab's local disk is temporary, so rerun this cell after a runtime reset.

In [ ]:
import zipfile

CONDITIONS = {'fog', 'night', 'rain', 'snow'}
SPLITS = {'train', 'val'}

def safe_extract_member(archive, member, destination):
    target = (destination / member.filename).resolve()
    if not target.is_relative_to(destination.resolve()):
        raise ValueError(f'Unsafe path in dataset archive: {member.filename}')
    archive.extract(member, destination)

ACDC_ROOT.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(ANNOTATIONS_ZIP) as archive:
    bad_member = archive.testzip()
    if bad_member:
        raise ValueError(f'Corrupt member in annotation archive: {bad_member}')
    annotation_members = [member for member in archive.infolist() if not member.is_dir()]
    missing_annotations = [
        member for member in annotation_members
        if not (ACDC_ROOT / member.filename).is_file()
    ]
    for member in missing_annotations:
        safe_extract_member(archive, member, ACDC_ROOT)
    print(f'Annotation files present: {len(annotation_members) - len(missing_annotations)} / {len(annotation_members)}')

with zipfile.ZipFile(IMAGES_ZIP) as archive:
    selected_members = []
    for member in archive.infolist():
        if member.is_dir():
            continue
        parts = Path(member.filename).parts
        if (
            'rgb_anon' in parts
            and CONDITIONS.intersection(parts)
            and SPLITS.intersection(parts)
        ):
            selected_members.append(member)
    if not selected_members:
        raise ValueError(
            'No RGB train/val files found in the image archive. '
            'Check that this is rgb_anon_trainvaltest.zip.'
        )
    missing_images = [
        member for member in selected_members
        if not (ACDC_ROOT / member.filename).is_file()
    ]
    print(f'Train/validation images present: {len(selected_members) - len(missing_images)} / {len(selected_members)}')
    print(f'Extracting {len(missing_images)} remaining files from the RGB archive...')
    for member in missing_images:
        safe_extract_member(archive, member, ACDC_ROOT)

annotation_marker = 'instancesonly_fog_train_gt_detection.json'
annotation_dirs = [path.parent.parent for path in ACDC_ROOT.rglob(annotation_marker)]
image_dirs = [path for path in ACDC_ROOT.rglob('rgb_anon') if path.is_dir()]
if len(annotation_dirs) != 1 or len(image_dirs) != 1:
    raise FileNotFoundError(
        'Could not identify one ACDC root containing gt_detection and rgb_anon. '
        f'Found {len(annotation_dirs)} annotation roots and {len(image_dirs)} image roots.'
    )

common_root = Path(__import__('os').path.commonpath([annotation_dirs[0], image_dirs[0]]))
if not (common_root / 'gt_detection').is_dir() or not (common_root / 'rgb_anon').is_dir():
    raise FileNotFoundError(f'Extracted data does not have the expected ACDC layout under {common_root}.')
ACDC_ROOT = common_root

required_annotations = [
    ACDC_ROOT / 'gt_detection' / condition / f'instancesonly_{condition}_{split}_gt_detection.json'
    for condition in sorted(CONDITIONS)
    for split in sorted(SPLITS)
]
missing = [path for path in required_annotations if not path.is_file()]
if missing:
    raise FileNotFoundError('Missing expected annotation files:\n' + '\n'.join(map(str, missing)))

print('ACDC root:', ACDC_ROOT)
print('Colab disk space after extraction:', shutil.disk_usage('/content').free // (1024 ** 3), 'GiB free')

## Convert and train

This uses the repository's converter and trainer. The converter generates class names from the ACDC annotation files and combines the four adverse-weather conditions into one detection dataset. Training output streams into this cell; at the end of each epoch, `last.pt`, `best.pt`, and `results.csv` are synchronized to Drive. If Colab disconnects, rerun the notebook and the training cell resumes from the latest completed epoch.

In [ ]:
conversion = subprocess.run(
    [
        sys.executable,
        str(PROJECT_DIR / 'dataset' / 'run_conversion.py'),
        '--acdc-root',
        str(ACDC_ROOT),
        '--output-dir',
        str(PROCESSED_DIR),
    ],
    cwd=PROJECT_DIR,
    text=True,
    capture_output=True,
)
if conversion.stdout:
    print(conversion.stdout)
if conversion.stderr:
    print(conversion.stderr, file=sys.stderr)
if conversion.returncode:
    raise RuntimeError(
        f'ACDC conversion failed with exit code {conversion.returncode}. '
        'Read the converter error printed above; do not start training until conversion succeeds.'
    )

DRIVE_CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
last_checkpoint = DRIVE_CHECKPOINT_DIR / 'last.pt'
training_command = [
    sys.executable,
    str(PROJECT_DIR / 'dataset' / 'yolo_train.py'),
    '--dataset-dir',
    str(PROCESSED_DIR),
    '--epochs',
    '50',
    '--imgsz',
    '640',
    '--backup-dir',
    str(DRIVE_CHECKPOINT_DIR),
]
if last_checkpoint.is_file() and not TRAINING_COMPLETE_MARKER.is_file():
    print(f'Resuming interrupted training from {last_checkpoint}', flush=True)
    training_command.extend(['--model', str(last_checkpoint), '--resume'])
else:
    if TRAINING_COMPLETE_MARKER.is_file():
        print('Previous run completed; starting a new training run.', flush=True)
        TRAINING_COMPLETE_MARKER.unlink()
    else:
        print('Starting a new training run.', flush=True)
    training_command.extend(['--model', 'yolov8n.pt'])

print('Training started. Batch output and an epoch/checkpoint update will appear below.', flush=True)
training_process = subprocess.Popen(
    training_command,
    cwd=PROJECT_DIR,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    bufsize=0,
)
try:
    if training_process.stdout is None:
        raise RuntimeError('Could not capture training output.')
    decoder = codecs.getincrementaldecoder('utf-8')('replace')
    while True:
        output_chunk = os.read(training_process.stdout.fileno(), 4096)
        if not output_chunk:
            break
        sys.stdout.write(decoder.decode(output_chunk))
        sys.stdout.flush()
    sys.stdout.write(decoder.decode(b'', final=True))
    sys.stdout.flush()
    return_code = training_process.wait()
except KeyboardInterrupt:
    training_process.terminate()
    training_process.wait()
    raise
if return_code:
    raise RuntimeError(
        f'Training exited with status {return_code}. The last completed epoch checkpoint, '
        f'if available, is saved in {DRIVE_CHECKPOINT_DIR}; rerun this cell to resume.'
    )
TRAINING_COMPLETE_MARKER.write_text('Training completed successfully.\n', encoding='utf-8')
print(f'Training complete. Drive checkpoints are in {DRIVE_CHECKPOINT_DIR}', flush=True)

## Save results to Drive

Copy the best model checkpoint, validation metrics, and generated class config to `datasets/ACDC/training_output/`. These files persist after the Colab runtime is removed.

In [ ]:
run_candidates = [
    path for path in (PROCESSED_DIR / 'runs').glob('acdc*')
    if path.is_dir() and (path / 'weights' / 'best.pt').is_file()
]
if not run_candidates:
    raise FileNotFoundError(f'No completed ACDC training run found under {PROCESSED_DIR / "runs"}.')
local_run_dir = max(run_candidates, key=lambda path: path.stat().st_mtime)
best_checkpoint = local_run_dir / 'weights' / 'best.pt'
if not best_checkpoint.is_file():
    raise FileNotFoundError(f'Training did not produce the expected checkpoint: {best_checkpoint}')

DRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
artifacts = [
    best_checkpoint,
    local_run_dir / 'results.csv',
    PROCESSED_DIR / 'data.yaml',
]
for artifact in artifacts:
    if artifact.is_file():
        destination = DRIVE_OUTPUT_DIR / artifact.name
        shutil.copy2(artifact, destination)
        print('Saved:', destination)
    elif artifact.name != 'results.csv':
        raise FileNotFoundError(f'Expected training artifact is missing: {artifact}')

print('Training artifacts saved in:', DRIVE_OUTPUT_DIR)